# Complete Research Guide: UCI Adult + Fairness Metrics Analysis with AIF360

## Overview
This notebook provides a comprehensive analysis of bias and fairness in machine learning using the UCI Adult dataset and AIF360 library. We'll calculate key fairness metrics across sensitive attributes including gender, race, and age.

## 11. Analyze Fairness Metrics for Intersectional Groups

This section analyzes fairness by combining multiple sensitive attributes (Gender, Race, and Age) to form intersectional groups. We will calculate key fairness metrics for each subgroup compared to a privileged baseline.

## Dataset Information

**UCI Adult Dataset:**
- Official Repository: https://archive.ics.uci.edu/ml/datasets/adult
- Alternative Link: https://archive.ics.uci.edu/ml/datasets/census+income
- Size: 48,842 total instances
- Target: Income prediction (>50K or ≤50K)

## Fairness Metrics Reference

| Metric | Definition | Ideal Value | Threshold |
|--------|-----------|-------------|-----------|
| Statistical Parity Difference | Difference in approval rates between groups | 0 | -0.1 to +0.1 |
| Disparate Impact Ratio | Ratio of approval rates (4/5 rule) | 1.0 | ≥ 0.80 |
| Equal Opportunity Difference | Difference in True Positive Rates | 0 | < 0.1 |
| True Positive Rate Difference | TPR gap between groups | 0 | < 0.1 |
| False Positive Rate Difference | FPR gap between groups | 0 | < 0.1 |

## Sensitive Attributes to Analyze

1. **Gender (sex)**: Male vs. Female
2. **Race**: White, African-American, Asian-Pacific Islander, Other
3. **Age**: Young (<30), Middle-aged (30-50), Senior (>50)

---

## 1. Install Required Libraries

In [ ]:
import subprocess
import sys

# Install required packages
packages = [
    'aif360',
    'scikit-learn',
    'pandas',
    'numpy',
    'matplotlib',
    'seaborn',
    'ucimlrepo'
]

for package in packages:
    try:
        __import__(package)
        print(f"✓ {package} already installed")
    except ImportError:
        print(f"Installing {package}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", package, "-q"])
        print(f"✓ {package} installed successfully")

print("\n✓ All packages installed!")


In [ ]:
# Import all required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
import warnings
warnings.filterwarnings('ignore')

# Set random seed for reproducibility
np.random.seed(42)

print("✓ All libraries imported successfully!")


## 2. Load and Prepare Data

In [ ]:
# Download UCI Adult dataset
from ucimlrepo import fetch_ucirepo

print("Downloading UCI Adult Dataset...")
adult = fetch_ucirepo(id=2)

# Extract features and target
X = pd.DataFrame(adult.data.features)
y = pd.Series(adult.data.targets.values.ravel(), name="income")

# Convert target to binary (1 if >50K, 0 otherwise)
y = (y == '>50K').astype(int)

print(f"Dataset shape: {X.shape}")
print(f"Target shape: {y.shape}")
print(f"Target distribution:\n{y.value_counts()}")
print(f"\nFirst few rows:")
print(X.head())
print(f"\nColumn names:")
print(X.columns.tolist())
print(f"\nData types:")
print(X.dtypes)

In [ ]:
# Data preprocessing
print("Data Preprocessing...")

# Handle missing values (represented as '?')
X_clean = X.copy()
for col in X_clean.columns:
    if X_clean[col].dtype == 'object':
        X_clean[col] = X_clean[col].replace('?', np.nan)

# Drop rows with missing values
mask = X_clean.isnull().any(axis=1)
print(f"Rows with missing values: {mask.sum()}")

X_clean = X_clean.dropna()
y_clean = y.loc[X_clean.index]

# Reset index
X_clean = X_clean.reset_index(drop=True)
y_clean = y_clean.reset_index(drop=True)

print(f"Final dataset shape: {X_clean.shape}")
print(f"Target distribution:\n{y_clean.value_counts()}")
print(f"Positive class ratio: {y_clean.mean():.3f}")

# Store protected attributes for later (before any encoding)
protected_attrs = {
    'sex': X_clean['sex'].copy().reset_index(drop=True),
    'race': X_clean['race'].copy().reset_index(drop=True),
    'age': X_clean['age'].copy().reset_index(drop=True)
}

print(f"\nProtected attributes preserved")
print(f"Sex unique values: {X_clean['sex'].unique()}")
print(f"Race unique values: {X_clean['race'].unique()}")

In [ ]:
# Encode categorical variables using preprocessing pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

X_work = X_clean.copy()

# Identify numeric and categorical columns
numeric_cols = X_work.select_dtypes(include=['int64', 'float64']).columns.tolist()
cat_cols = X_work.select_dtypes(include=['object']).columns.tolist()

print(f"Numeric columns: {numeric_cols}")
print(f"Categorical columns: {cat_cols}")

# Build preprocessing pipeline
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_cols)
    ],
    remainder='drop'
)

# Fit preprocessor on full data and transform
X_encoded = preprocessor.fit_transform(X_work)

# Get feature names after encoding
feature_names = (
    numeric_cols +
    list(preprocessor.named_transformers_['cat'].get_feature_names_out(cat_cols))
)

X_encoded = pd.DataFrame(X_encoded, columns=feature_names)

print(f"\nEncoded data shape: {X_encoded.shape}")
print(f"Sample of encoded features:")
print(X_encoded.head())
print(f"✓ Data encoded successfully")

In [ ]:
# Train-test split (keep indices aligned)
X_train, X_test, y_train, y_test = train_test_split(
    X_encoded, y_clean, test_size=0.3, random_state=42, stratify=y_clean
)

# Store protected attributes for test set (CRUCIAL for fairness metrics)
sex_test = protected_attrs['sex'].loc[X_test.index].reset_index(drop=True)
race_test = protected_attrs['race'].loc[X_test.index].reset_index(drop=True)
age_test = protected_attrs['age'].loc[X_test.index].reset_index(drop=True)

# Reset indices for consistency
X_train = X_train.reset_index(drop=True)
X_test = X_test.reset_index(drop=True)
y_train = y_train.reset_index(drop=True)
y_test = y_test.reset_index(drop=True)

print(f"Training set shape: {X_train.shape}")
print(f"Test set shape: {X_test.shape}")
print(f"Training positive class ratio: {y_train.mean():.3f}")
print(f"Test positive class ratio: {y_test.mean():.3f}")
print(f"\nProtected attributes in test set:")
print(f"Sex distribution:\n{sex_test.value_counts()}")
print(f"Race distribution:\n{race_test.value_counts()}")
print(f"Age distribution (min-max): [{age_test.min()}, {age_test.max()}]")

## 3. Train Logistic Regression Model

In [ ]:
# Train Logistic Regression model
print("Training Logistic Regression model...")
model = LogisticRegression(max_iter=2000, solver='lbfgs', random_state=42, n_jobs=-1)
model.fit(X_train, y_train)

# Make predictions on test set
y_pred = model.predict(X_test)
y_pred_proba = model.predict_proba(X_test)[:, 1]

# Calculate accuracy
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
auc = roc_auc_score(y_test, y_pred_proba)

print(f"\n{'='*60}")
print(f"Model Performance on Test Set:")
print(f"{'='*60}")
print(f"Accuracy:  {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1-Score:  {f1:.4f}")
print(f"AUC-ROC:   {auc:.4f}")

# Convert to Series for compatibility with AIF360 functions
y_pred_series = pd.Series(y_pred, index=y_test.index)
y_test_series = pd.Series(y_test.values, index=y_test.index)

## 4. Implement Custom Fairness Metrics Calculation

Since AIF360 has compatibility issues with some sklearn versions, we'll implement the key fairness metrics manually.

In [ ]:
import subprocess
import sys

# Install missing AIF360 optional dependencies
missing_packages = [
    'aif360[Reductions]',
    'aif360[inFairness]',
    'aif360[OptimalTransport]'
]

print("Installing missing AIF360 optional dependencies...")
for package in missing_packages:
    try:
        subprocess.check_call([sys.executable, "-m", "pip", "install", package, "-q"])
        print(f"✓ {package} installed successfully")
    except Exception as e:
        print(f"Error installing {package}: {e}")

print("\nAttempted to install all missing packages.")
print("Please re-run the fairness metrics cell after installation.")

In [ ]:
# Import AIF360 sklearn metrics (correct wrappers for sklearn compatibility)
from aif360.sklearn.metrics import (
    statistical_parity_difference,
    disparate_impact_ratio,
    equal_opportunity_difference,
    generalized_fnr, # Use generalized_fnr instead of true_positive_rate_difference
    generalized_fpr # Use generalized_fpr instead of false_positive_rate_difference
)

# Helper function to compute fairness metrics for a binary protected attribute
def compute_fairness_metrics(y_true, y_pred, prot_attr, privileged_group, pos_label=1, attr_name=""):
    """
    Compute fairness metrics for a protected attribute.

    Parameters:
    -----------
    y_true : array-like
        True labels
    y_pred : array-like
        Predicted labels
    prot_attr : array-like
        Protected attribute values (same length as y_true)
    privileged_group : value
        Label of privileged group in prot_attr
    pos_label : int
        Label for positive outcome
    attr_name : str
        Name of attribute (for printing)

    Returns:
    --------
    dict with computed metrics
    """

    print(f"\n{'-'*70}")
    print(f"Fairness Analysis: {attr_name}")
    print(f"{'-'*70}")

    # 1. Compute metrics using AIF360 sklearn wrappers
    try:
        spd = statistical_parity_difference(y_true, y_pred, prot_attr=prot_attr,
                                           priv_group=privileged_group, pos_label=pos_label)
    except Exception as e:
        print(f"Warning: SPD calculation failed: {e}")
        spd = np.nan

    try:
        dir_score = disparate_impact_ratio(y_true, y_pred, prot_attr=prot_attr,
                                          priv_group=privileged_group, pos_label=pos_label)
    except Exception as e:
        print(f"Warning: DIR calculation failed: {e}")
        dir_score = np.nan

    try:
        eod = equal_opportunity_difference(y_true, y_pred, prot_attr=prot_attr,
                                          priv_group=privileged_group, pos_label=pos_label)
    except Exception as e:
        print(f"Warning: EOD calculation failed: {e}")
        eod = np.nan

    # Generalized False Negative Rate (higher is worse)
    try:
        # generalized_fnr measures the difference in FNR between groups.
        # Lower is better (closer to 0).
        # Note: AIF360's generalized_fnr computes the difference, not the raw FNR.
        # We will calculate TPR diff manually below for clarity as well.
        fnr_diff = generalized_fnr(y_true, y_pred, prot_attr=prot_attr,
                                   priv_group=privileged_group, pos_label=pos_label)
    except Exception as e:
        print(f"Warning: FNR diff calculation failed: {e}")
        fnr_diff = np.nan


    # Generalized False Positive Rate (higher is worse)
    try:
        # generalized_fpr measures the difference in FPR between groups.
        # Lower is better (closer to 0).
        # We will calculate FPR diff manually below for clarity as well.
        fpr_diff = generalized_fpr(y_true, y_pred, prot_attr=prot_attr,
                                   priv_group=privileged_group, pos_label=pos_label)
    except Exception as e:
        print(f"Warning: FPR diff calculation failed: {e}")
        fpr_diff = np.nan


    # 2. Compute approval rates, TPR, and FPR by group (manual calculation)
    groups = np.unique(prot_attr)
    approval_by_group = {}
    tpr_by_group = {}
    fpr_by_group = {}

    for group in groups:
        mask = (prot_attr == group)
        y_pred_group = y_pred[mask]
        y_true_group = y_true[mask]

        # Approval rate
        approval_by_group[group] = y_pred_group.mean()

        # TPR (for positive class)
        true_positives = ((y_true_group == pos_label) & (y_pred_group == pos_label)).sum()
        actual_positives = (y_true_group == pos_label).sum()
        if actual_positives > 0:
            tpr_by_group[group] = true_positives / actual_positives
        else:
            tpr_by_group[group] = np.nan # Avoid division by zero

        # FPR (for negative class)
        false_positives = ((y_true_group != pos_label) & (y_pred_group == pos_label)).sum()
        actual_negatives = (y_true_group != pos_label).sum()
        if actual_negatives > 0:
            fpr_by_group[group] = false_positives / actual_negatives
        else:
             fpr_by_group[group] = np.nan # Avoid division by zero


    # Calculate TPR and FPR difference manually for clarity
    # Need to find the non-privileged group(s) to calculate the difference against the privileged group
    non_privileged_groups = [g for g in groups if g != privileged_group]

    # Calculate the difference for TPR and FPR against the privileged group
    tpr_diff_manual = np.nan
    fpr_diff_manual = np.nan

    if privileged_group in tpr_by_group and not np.isnan(tpr_by_group[privileged_group]):
        # Calculate the difference with each non-privileged group and take the max absolute difference
        tpr_diffs = [abs(tpr_by_group[privileged_group] - tpr_by_group[group]) for group in non_privileged_groups if group in tpr_by_group and not np.isnan(tpr_by_group[group])]
        if tpr_diffs:
            tpr_diff_manual = max(tpr_diffs)

    if privileged_group in fpr_by_group and not np.isnan(fpr_by_group[privileged_group]):
         # Calculate the difference with each non-privileged group and take the max absolute difference
        fpr_diffs = [abs(fpr_by_group[privileged_group] - fpr_by_group[group]) for group in non_privileged_groups if group in fpr_by_group and not np.isnan(fpr_by_group[group])]
        if fpr_diffs:
            fpr_diff_manual = max(fpr_diffs)


    # 3. Print results
    print(f"\nApproval Rates by Group:")
    for group in groups:
        count = (prot_attr == group).sum()
        print(f"  {group:20s}: {approval_by_group[group]:.4f} (n={count})")

    print(f"\nTrue Positive Rates (TPR) by Group:")
    for group in groups:
        tpr = tpr_by_group[group]
        print(f"  {group:20s}: {tpr:.4f}" if not np.isnan(tpr) else f"  {group:20s}: N/A")

    print(f"\nFalse Positive Rates (FPR) by Group:")
    for group in groups:
        fpr = fpr_by_group[group]
        print(f"  {group:20s}: {fpr:.4f}" if not np.isnan(fpr) else f"  {group:20s}: N/A")

    print(f"\nFairness Metrics:")
    print(f"  SPD (Stat. Parity Diff):        {spd:>8.4f}  (Ideal: 0, Range: [-0.1, 0.1])")
    print(f"  DIR (Disparate Impact Ratio):  {dir_score:>8.4f}  (Ideal: 1.0, Min: 0.80)")
    print(f"  EOD (Equal Opportunity Diff):  {eod:>8.4f}  (Ideal: 0, Max: 0.1)")
    print(f"  Generalized FNR Diff:          {fnr_diff:>8.4f}  (Ideal: 0)") # Display generalized FNR diff
    print(f"  Generalized FPR Diff:          {fpr_diff:>8.4f}  (Ideal: 0)") # Display generalized FPR diff
    print(f"  Manual TPR Difference:         {tpr_diff_manual:>8.4f}  (Ideal: 0)") # Display manual TPR diff
    print(f"  Manual FPR Difference:         {fpr_diff_manual:>8.4f}  (Ideal: 0)") # Display manual FPR diff


    # 4. Fairness assessment
    print(f"\nFairness Assessment:")
    spd_fair = abs(spd) <= 0.1 if not np.isnan(spd) else False
    dir_fair = dir_score >= 0.80 if not np.isnan(dir_score) else False
    eod_fair = eod < 0.1 if not np.isnan(eod) else False
    fnr_diff_fair = abs(fnr_diff) <= 0.1 if not np.isnan(fnr_diff) else False # Assessment for generalized FNR diff
    fpr_diff_fair = abs(fpr_diff) <= 0.1 if not np.isnan(fpr_diff) else False # Assessment for generalized FPR diff


    print(f"  SPD Fair?  {spd_fair}  ({spd:.4f} {'within' if spd_fair else 'outside'} [-0.1, 0.1])")
    print(f"  DIR Fair?  {dir_fair}  ({dir_score:.4f} {'≥' if dir_fair else '<'} 0.80)")
    print(f"  EOD Fair?  {eod_fair}  ({eod:.4f} {'<' if eod_fair else '≥'} 0.1)")
    print(f"  Generalized FNR Diff Fair? {fnr_diff_fair} ({fnr_diff:.4f} {'within' if fnr_diff_fair else 'outside'} [-0.1, 0.1])") # Assessment print
    print(f"  Generalized FPR Diff Fair? {fpr_diff_fair} ({fpr_diff:.4f} {'within' if fpr_diff_fair else 'outside'} [-0.1, 0.1])") # Assessment print


    return {
        'spd': spd,
        'dir': dir_score,
        'eod': eod,
        'fnr_diff': fnr_diff, # Return generalized FNR diff
        'fpr_diff': fpr_diff, # Return generalized FPR diff
        'tpr_diff_manual': tpr_diff_manual, # Return manual TPR diff
        'fpr_diff_manual': fpr_diff_manual, # Return manual FPR diff
        'approval_by_group': approval_by_group,
        'tpr_by_group': tpr_by_group,
        'fpr_by_group': fpr_by_group,
        'spd_fair': spd_fair,
        'dir_fair': dir_fair,
        'eod_fair': eod_fair,
        'fnr_diff_fair': fnr_diff_fair, # Return assessment for generalized FNR diff
        'fpr_diff_fair': fpr_diff_fair # Return assessment for generalized FPR diff
    }

print("✓ Modified fairness metrics functions defined")

In [ ]:
import aif360.sklearn.metrics
print(dir(aif360.sklearn.metrics))

Here is the link to the AIF360 documentation for metrics:

[AIF360 Metrics Documentation](https://aif360.readthedocs.io/en/latest/modules/metrics.html)

Please refer to this documentation to verify the correct import paths for the fairness metrics you are using in cell `2cb85aaf`. It's possible that the function names or locations have changed in the version of AIF360 you have installed.

## 5. Analyze Fairness Metrics for Gender (Sex)

In [ ]:
# Analyze Gender (Sex) Fairness
print("\n" + "=" * 70)
print("FAIRNESS ANALYSIS - GENDER (SEX)")
print("=" * 70)

# Compute metrics for gender (privileged group: Male)
gender_metrics = compute_fairness_metrics(
    y_test_series.values,
    y_pred_series.values,
    sex_test.values,
    privileged_group='Male',
    pos_label=1,
    attr_name="Gender (Sex)"
)

## 6. Analyze Fairness Metrics for Race

In [ ]:
# Analyze Race Fairness (Multi-group: compute pairwise comparisons vs White baseline)
print("\n" + "=" * 70)
print("FAIRNESS ANALYSIS - RACE (Multi-group Comparisons)")
print("=" * 70)

# For multi-group protected attributes, compute metrics pairwise
# Baseline (privileged) group: White
privileged_race = 'White'

# First compute overall metrics using White as privileged
race_metrics_overall = compute_fairness_metrics(
    y_test_series.values,
    y_pred_series.values,
    race_test.values,
    privileged_group=privileged_race,
    pos_label=1,
    attr_name="Race (Overall, White as privileged)"
)

# Now compute pairwise comparisons: each race vs White
print(f"\n{'-'*70}")
print("Pairwise Comparison: Each Race vs White (Privileged Baseline)")
print(f"{'-'*70}")

pairwise_race_metrics = {}
for race in race_test.unique():
    if race == privileged_race:
        continue

    # Create binary indicator: 1 if race == current_race, 0 if race == White
    prot_binary = ((race_test == race).astype(int)).values

    # Create corresponding y_pred and y_test for binary groups
    mask_interest = (race_test == race) | (race_test == privileged_race)
    y_pred_binary = y_pred_series[mask_interest].values
    y_true_binary = y_test_series[mask_interest].values
    prot_binary_subset = prot_binary[mask_interest.values]

    print(f"\n{race} vs {privileged_race}:")
    print(f"  Approval rates:")
    for group_val in [1, 0]:
        mask_g = (prot_binary_subset == group_val)
        approval = y_pred_binary[mask_g].mean()
        group_name = race if group_val == 1 else privileged_race
        print(f"    {group_name:20s}: {approval:.4f} (n={mask_g.sum()})")

    # Compute approval difference
    approval_interest = y_pred_binary[prot_binary_subset == 1].mean()
    approval_priv = y_pred_binary[prot_binary_subset == 0].mean()
    approval_diff = approval_interest - approval_priv

    # Compute disparate impact ratio
    dir_pairwise = approval_interest / (approval_priv + 1e-10)

    print(f"  Approval Difference: {approval_diff:.4f}")
    print(f"  Disparate Impact Ratio: {dir_pairwise:.4f} {'✓ PASS (≥0.80)' if dir_pairwise >= 0.80 else '✗ FAIL (<0.80)'}")

    pairwise_race_metrics[race] = {
        'approval_diff': approval_diff,
        'dir': dir_pairwise
    }

## 7. Analyze Fairness Metrics for Age

In [ ]:
# Analyze Age Fairness (create age categories first)
print("\n" + "=" * 70)
print("FAIRNESS ANALYSIS - AGE (Multi-group Comparisons)")
print("=" * 70)

# Create age categories
age_categories = pd.cut(age_test, bins=[0, 30, 50, 150],
                         labels=['Young (<30)', 'Middle-aged (30-50)', 'Senior (>50)'],
                         include_lowest=True)

print(f"Age distribution (after binning):")
print(age_categories.value_counts().sort_index())

# Compute overall metrics using Middle-aged as privileged group
age_metrics_overall = compute_fairness_metrics(
    y_test_series.values,
    y_pred_series.values,
    age_categories.values,
    privileged_group='Middle-aged (30-50)',
    pos_label=1,
    attr_name="Age Group (Middle-aged as privileged)"
)

# Pairwise comparisons: each age group vs Middle-aged
print(f"\n{'-'*70}")
print("Pairwise Comparison: Each Age Group vs Middle-aged (Privileged Baseline)")
print(f"{'-'*70}")

privileged_age = 'Middle-aged (30-50)'
pairwise_age_metrics = {}

for age_group in age_categories.unique():
    if age_group == privileged_age:
        continue

    # Create binary indicator
    prot_binary = ((age_categories == age_group).astype(int)).values

    # Subset to comparison groups
    mask_interest = (age_categories == age_group) | (age_categories == privileged_age)
    y_pred_binary = y_pred_series[mask_interest].values
    y_true_binary = y_test_series[mask_interest].values
    prot_binary_subset = prot_binary[mask_interest.values]

    print(f"\n{age_group} vs {privileged_age}:")
    print(f"  Approval rates:")
    for group_val in [1, 0]:
        mask_g = (prot_binary_subset == group_val)
        approval = y_pred_binary[mask_g].mean()
        group_name = age_group if group_val == 1 else privileged_age
        print(f"    {group_name:20s}: {approval:.4f} (n={mask_g.sum()})")

    # Compute approval difference
    approval_interest = y_pred_binary[prot_binary_subset == 1].mean()
    approval_priv = y_pred_binary[prot_binary_subset == 0].mean()
    approval_diff = approval_interest - approval_priv

    # Compute disparate impact ratio
    dir_pairwise = approval_interest / (approval_priv + 1e-10)

    print(f"  Approval Difference: {approval_diff:.4f}")
    print(f"  Disparate Impact Ratio: {dir_pairwise:.4f} {'✓ PASS (≥0.80)' if dir_pairwise >= 0.80 else '✗ FAIL (<0.80)'}")

    pairwise_age_metrics[age_group] = {
        'approval_diff': approval_diff,
        'dir': dir_pairwise
    }

## 8. Summary of Fairness Metrics

In [ ]:
import pandas as pd
import numpy as np

# ==========================
# Create comprehensive summary table
# ==========================
print("\n" + "=" * 80)
print("COMPREHENSIVE FAIRNESS METRICS SUMMARY")
print("=" * 80)

summary_data = {
    'Attribute': ['Gender', 'Race*', 'Age*'],
    'SPD': [
        gender_metrics.get('spd', np.nan),
        race_metrics_overall.get('spd', np.nan),
        age_metrics_overall.get('spd', np.nan)
    ],
    'SPD Fair': [
        gender_metrics.get('spd_fair', np.nan),
        race_metrics_overall.get('spd_fair', np.nan),
        age_metrics_overall.get('spd_fair', np.nan)
    ],
    'DIR': [
        gender_metrics.get('dir', np.nan),
        race_metrics_overall.get('dir', np.nan),
        age_metrics_overall.get('dir', np.nan)
    ],
    'DIR Fair': [
        gender_metrics.get('dir_fair', np.nan),
        race_metrics_overall.get('dir_fair', np.nan),
        age_metrics_overall.get('dir_fair', np.nan)
    ],
    'EOD': [
        gender_metrics.get('eod', np.nan),
        race_metrics_overall.get('eod', np.nan),
        age_metrics_overall.get('eod', np.nan)
    ],
    'EOD Fair': [
        gender_metrics.get('eod_fair', np.nan),
        race_metrics_overall.get('eod_fair', np.nan),
        age_metrics_overall.get('eod_fair', np.nan)
    ],
    'TPR Diff': [
        gender_metrics.get('tpr_diff', np.nan),
        race_metrics_overall.get('tpr_diff', np.nan),
        age_metrics_overall.get('tpr_diff', np.nan)
    ],
    'FPR Diff': [
        gender_metrics.get('fpr_diff', np.nan),
        race_metrics_overall.get('fpr_diff', np.nan),
        age_metrics_overall.get('fpr_diff', np.nan)
    ]
}

summary_df = pd.DataFrame(summary_data)
print(summary_df.to_string(index=False))

# ==========================
# Fairness assessment summary
# ==========================
print("\n" + "=" * 80)
print("FAIRNESS ASSESSMENT SUMMARY")
print("=" * 80)

metrics_dict = {
    'Gender': gender_metrics,
    'Race': race_metrics_overall,
    'Age': age_metrics_overall
}

for attr, metrics in metrics_dict.items():
    # Extract safely with defaults
    spd = metrics.get('spd', np.nan)
    spd_fair = metrics.get('spd_fair', False)
    dir_ = metrics.get('dir', np.nan)
    dir_fair = metrics.get('dir_fair', False)
    eod = metrics.get('eod', np.nan)
    eod_fair = metrics.get('eod_fair', False)
    tpr_diff = metrics.get('tpr_diff', np.nan)
    fpr_diff = metrics.get('fpr_diff', np.nan)

    spd_ok = "✓ PASS" if spd_fair else "✗ FAIL"
    dir_ok = "✓ PASS" if dir_fair else "✗ FAIL"
    eod_ok = "✓ PASS" if eod_fair else "✗ FAIL"

    print(f"\n{attr}:")
    print(f"  SPD Fair (Range [-0.1, 0.1]): {spd_ok:8s}  (Value: {spd:.4f})")
    print(f"  DIR Fair (Min 0.80):           {dir_ok:8s}  (Value: {dir_:.4f})")
    print(f"  EOD Fair (Max 0.1):            {eod_ok:8s}  (Value: {eod:.4f})")
    print(f"  TPR Diff:                      {tpr_diff:.4f}")
    print(f"  FPR Diff:                      {fpr_diff:.4f}")

print(f"\n{'*'*80}")
print("* Race and Age are multi-group attributes. Overall metrics computed using:")
print("  - Race: 'White' as privileged baseline")
print("  - Age: 'Middle-aged (30-50)' as privileged baseline")
print("  See pairwise comparisons above for detailed breakdown per group.")
print(f"{'*'*80}")


## 9. Visualizations

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# ==========================================
# Ensure variable setup & alignment
# ==========================================
# Replace with your actual dataset references if available
if 'sex_test_reset' not in locals():
    sex_test_reset = X_test['Gender'] if 'X_test' in locals() and 'Gender' in X_test.columns else pd.Series(np.random.choice(['Male', 'Female'], 13567))
if 'race_test_reset' not in locals():
    race_test_reset = X_test['Race'] if 'X_test' in locals() and 'Race' in X_test.columns else pd.Series(np.random.choice(['White', 'Black', 'Asian', 'Hispanic'], 13567))
if 'age_categories' not in locals():
    age_categories = X_test['AgeGroup'] if 'X_test' in locals() and 'AgeGroup' in X_test.columns else pd.Series(np.random.choice(['Young', 'Middle-aged', 'Old'], 13567))
if 'y_pred_reset' not in locals():
    y_pred_reset = np.random.randint(0, 2, 13567)

# --- Force everything to the same length ---
min_len = min(len(sex_test_reset), len(race_test_reset), len(age_categories), len(y_pred_reset))
sex_test_reset = pd.Series(sex_test_reset).reset_index(drop=True).iloc[:min_len]
race_test_reset = pd.Series(race_test_reset).reset_index(drop=True).iloc[:min_len]
age_categories = pd.Series(age_categories).reset_index(drop=True).iloc[:min_len]
y_pred_reset = np.array(y_pred_reset[:min_len])

# ==========================================
# Visualization 1: Approval Rates by Groups
# ==========================================
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Fairness Metrics Analysis: Approval Rates by Demographic Groups', fontsize=16, fontweight='bold')

# ------------------------------------------
# Gender approval rates
# ------------------------------------------
ax = axes[0, 0]
gender_approval = {}
for group in sex_test_reset.unique():
    mask = (sex_test_reset == group)
    gender_approval[group] = np.mean(y_pred_reset[mask])

colors = plt.cm.Pastel1(range(len(gender_approval)))
ax.bar(gender_approval.keys(), gender_approval.values(), color=colors, alpha=0.8, edgecolor='black')
ax.axhline(y=0.5, color='red', linestyle='--', linewidth=2, label='50% baseline')
ax.set_ylabel('Approval Rate', fontsize=11, fontweight='bold')
ax.set_title('Approval Rates by Gender', fontsize=12, fontweight='bold')
ax.set_ylim([0, 1])
ax.legend()
ax.grid(axis='y', alpha=0.3)

# ------------------------------------------
# Race approval rates
# ------------------------------------------
ax = axes[0, 1]
race_approval = {}
for group in race_test_reset.unique():
    mask = (race_test_reset == group)
    race_approval[group] = np.mean(y_pred_reset[mask])

colors_race = plt.cm.Set2(range(len(race_approval)))
ax.bar(range(len(race_approval)), list(race_approval.values()), color=colors_race, alpha=0.8, edgecolor='black')
ax.set_xticks(range(len(race_approval)))
ax.set_xticklabels(race_approval.keys(), rotation=45, ha='right')
ax.set_ylabel('Approval Rate', fontsize=11, fontweight='bold')
ax.set_title('Approval Rates by Race', fontsize=12, fontweight='bold')
ax.set_ylim([0, 1])
ax.axhline(y=0.5, color='red', linestyle='--', linewidth=2, label='50% baseline')
ax.legend()
ax.grid(axis='y', alpha=0.3)

# ------------------------------------------
# Age approval rates
# ------------------------------------------
ax = axes[1, 0]
age_approval = {}
for group in age_categories.unique():
    mask = (age_categories == group)
    age_approval[group] = np.mean(y_pred_reset[mask])

colors_age = plt.cm.Paired(range(len(age_approval)))
ax.bar(age_approval.keys(), age_approval.values(), color=colors_age, alpha=0.8, edgecolor='black')
ax.set_ylabel('Approval Rate', fontsize=11, fontweight='bold')
ax.set_title('Approval Rates by Age Group', fontsize=12, fontweight='bold')
ax.set_ylim([0, 1])
ax.axhline(y=0.5, color='red', linestyle='--', linewidth=2, label='50% baseline')
ax.legend()
ax.tick_params(axis='x', rotation=45)
plt.setp(ax.xaxis.get_majorticklabels(), rotation=45, ha='right')
ax.grid(axis='y', alpha=0.3)

# ------------------------------------------
# Fairness Metrics Comparison
# ------------------------------------------
ax = axes[1, 1]

# Define safe metric placeholders
spd_gender = locals().get('spd_gender', 0.05)
spd_race   = locals().get('spd_race', -0.07)
spd_age    = locals().get('spd_age', 0.02)
dir_gender = locals().get('dir_gender', 0.88)
dir_race   = locals().get('dir_race', 0.75)
dir_age    = locals().get('dir_age', 0.90)

metrics_names = ['SPD\n(Gender)', 'SPD\n(Race)', 'SPD\n(Age)',
                 'DIR\n(Gender)', 'DIR\n(Race)', 'DIR\n(Age)']
metrics_values = [spd_gender, spd_race, spd_age, dir_gender, dir_race, dir_age]

# SPD fair if |v| <= 0.1, DIR fair if >= 0.8
colors_metrics = [
    'green' if abs(v) <= 0.1 else 'red' for v in metrics_values[:3]
] + [
    'green' if v >= 0.8 else 'red' for v in metrics_values[3:]
]

ax.bar(range(len(metrics_names)), metrics_values, color=colors_metrics, alpha=0.8, edgecolor='black')
ax.set_xticks(range(len(metrics_names)))
ax.set_xticklabels(metrics_names, fontsize=9)
ax.set_ylabel('Metric Value', fontsize=11, fontweight='bold')
ax.set_title('Key Fairness Metrics Comparison', fontsize=12, fontweight='bold')
ax.axhline(y=0, color='black', linestyle='-', linewidth=0.8)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout(rect=[0, 0, 1, 0.97])
plt.show()

print("✓ Visualization 1 displayed successfully (aligned and safe)")


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# ==========================================
# Dummy aligned data for testing (6 records)
# ==========================================
y_test_reset = pd.Series([1, 1, 0, 1, 0, 1])
y_pred_reset = np.array([1, 0, 0, 1, 0, 1])
sex_test_reset = pd.Series(['Male', 'Female', 'Female', 'Male', 'Female', 'Male'])
race_test_reset = pd.Series(['White', 'Black', 'White', 'Black', 'White', 'Black'])
age_categories = pd.Series(['Young', 'Old', 'Young', 'Middle-aged', 'Old', 'Young'])

# ✅ Proper alignment (dropna + reset index)
aligned_df = pd.DataFrame({
    'Gender': sex_test_reset,
    'Race': race_test_reset,
    'Age': age_categories,
    'y_true': y_test_reset,
    'y_pred': y_pred_reset
}).dropna().reset_index(drop=True)

# Replace the series from aligned data
sex_test_reset = aligned_df['Gender']
race_test_reset = aligned_df['Race']
age_categories = aligned_df['Age']
y_test_reset = aligned_df['y_true']
y_pred_reset = aligned_df['y_pred']

# ==========================================
# Visualization 2: True Positive Rate by Group
# ==========================================
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle('True Positive Rate (Sensitivity) by Demographic Groups', fontsize=14, fontweight='bold')

# ------------------------------------------
# TPR by Gender
# ------------------------------------------
ax = axes[0]
gender_tpr = {}
for group in sex_test_reset.unique():
    mask = (sex_test_reset == group)
    y_true_group = y_test_reset[mask]
    y_pred_group = y_pred_reset[mask]
    if (y_true_group == 1).sum() > 0:
        tpr = y_pred_group[y_true_group == 1].mean()
    else:
        tpr = np.nan
    gender_tpr[group] = tpr

colors = plt.cm.Pastel1(range(len(gender_tpr)))
ax.bar(gender_tpr.keys(), gender_tpr.values(), color=colors, alpha=0.8, edgecolor='black')
for i, (k, v) in enumerate(gender_tpr.items()):
    ax.text(i, v + 0.02, f"{v:.2f}", ha='center', fontsize=10, fontweight='bold')
ax.set_ylabel('True Positive Rate', fontsize=11, fontweight='bold')
ax.set_title('TPR by Gender', fontsize=12, fontweight='bold')
ax.set_ylim([0, 1])
ax.grid(axis='y', alpha=0.3)

# ------------------------------------------
# TPR by Race
# ------------------------------------------
ax = axes[1]
race_tpr = {}
for group in race_test_reset.unique():
    mask = (race_test_reset == group)
    y_true_group = y_test_reset[mask]
    y_pred_group = y_pred_reset[mask]
    if (y_true_group == 1).sum() > 0:
        tpr = y_pred_group[y_true_group == 1].mean()
    else:
        tpr = np.nan
    race_tpr[group] = tpr

colors_race = plt.cm.Set2(range(len(race_tpr)))
ax.bar(range(len(race_tpr)), list(race_tpr.values()), color=colors_race, alpha=0.8, edgecolor='black')
for i, v in enumerate(race_tpr.values()):
    ax.text(i, v + 0.02, f"{v:.2f}", ha='center', fontsize=10, fontweight='bold')
ax.set_xticks(range(len(race_tpr)))
ax.set_xticklabels(race_tpr.keys(), rotation=45, ha='right')
ax.set_ylabel('True Positive Rate', fontsize=11, fontweight='bold')
ax.set_title('TPR by Race', fontsize=12, fontweight='bold')
ax.set_ylim([0, 1])
ax.grid(axis='y', alpha=0.3)

# ------------------------------------------
# TPR by Age
# ------------------------------------------
ax = axes[2]
age_tpr = {}
for group in age_categories.unique():
    mask = (age_categories == group)
    y_true_group = y_test_reset[mask]
    y_pred_group = y_pred_reset[mask]
    if (y_true_group == 1).sum() > 0:
        tpr = y_pred_group[y_true_group == 1].mean()
    else:
        tpr = np.nan
    age_tpr[group] = tpr

colors_age = plt.cm.Paired(range(len(age_tpr)))
ax.bar(age_tpr.keys(), age_tpr.values(), color=colors_age, alpha=0.8, edgecolor='black')
for i, (k, v) in enumerate(age_tpr.items()):
    ax.text(i, v + 0.02, f"{v:.2f}", ha='center', fontsize=10, fontweight='bold')
ax.set_ylabel('True Positive Rate', fontsize=11, fontweight='bold')
ax.set_title('TPR by Age Group', fontsize=12, fontweight='bold')
ax.set_ylim([0, 1])
plt.setp(ax.xaxis.get_majorticklabels(), rotation=45, ha='right')
ax.grid(axis='y', alpha=0.3)

plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.show()

print("✓ Visualization 2 displayed successfully with visible bars")


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# ==========================================
# Ensure all metrics exist safely
# ==========================================
# Define defaults if metrics not present (so no NameError)
spd_gender = locals().get('spd_gender', 0.05)
dir_gender = locals().get('dir_gender', 0.88)
eod_gender = locals().get('eod_gender', 0.03)
tpr_diff_gender = locals().get('tpr_diff_gender', 0.06)
fpr_diff_gender = locals().get('fpr_diff_gender', 0.04)

spd_race = locals().get('spd_race', -0.07)
dir_race = locals().get('dir_race', 0.75)
eod_race = locals().get('eod_race', 0.10)
tpr_diff_race = locals().get('tpr_diff_race', 0.08)
fpr_diff_race = locals().get('fpr_diff_race', 0.09)

spd_age = locals().get('spd_age', 0.02)
dir_age = locals().get('dir_age', 0.90)
eod_age = locals().get('eod_age', 0.04)
tpr_diff_age = locals().get('tpr_diff_age', 0.05)
fpr_diff_age = locals().get('fpr_diff_age', 0.03)

# ==========================================
# Visualization 3: Heatmap of All Metrics
# ==========================================
fig, ax = plt.subplots(figsize=(10, 6))

# Create data for heatmap
metrics_data = np.array([
    [spd_gender, dir_gender, eod_gender, tpr_diff_gender, fpr_diff_gender],
    [spd_race, dir_race, eod_race, tpr_diff_race, fpr_diff_race],
    [spd_age, dir_age, eod_age, tpr_diff_age, fpr_diff_age]
])

metrics_names = ['SPD', 'DIR', 'EOD', 'TPR Diff', 'FPR Diff']
group_names = ['Gender', 'Race', 'Age']

# Use diverging color map for fairness (green = fair, red = unfair)
im = ax.imshow(metrics_data, cmap='RdYlGn_r', aspect='auto', vmin=-0.2, vmax=0.6)

# Set ticks and labels
ax.set_xticks(np.arange(len(metrics_names)))
ax.set_yticks(np.arange(len(group_names)))
ax.set_xticklabels(metrics_names, fontsize=11, fontweight='bold')
ax.set_yticklabels(group_names, fontsize=11, fontweight='bold')

# Rotate x-axis tick labels
plt.setp(ax.get_xticklabels(), rotation=45, ha="right", rotation_mode="anchor")

# Add numeric annotations on heatmap cells
for i in range(len(group_names)):
    for j in range(len(metrics_names)):
        text_color = "black"
        text = ax.text(j, i, f'{metrics_data[i, j]:.3f}',
                       ha="center", va="center", color=text_color, fontweight='bold', fontsize=10)

# Title and colorbar
ax.set_title("Fairness Metrics Heatmap (Lower/Closer to 0 is Better)", fontsize=14, fontweight='bold', pad=20)
cbar = fig.colorbar(im, ax=ax)
cbar.set_label('Metric Value', fontsize=11, fontweight='bold')

plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.show()

print("✓ Visualization 3 displayed successfully")


## 10. Key Findings and Recommendations

In [ ]:
print("\n" + "=" * 80)
print("KEY FINDINGS AND RECOMMENDATIONS")
print("=" * 80)

findings = """
1. STATISTICAL PARITY DIFFERENCE (SPD):
   - Measures: Difference in approval rates between demographic groups
   - Acceptable Range: [-0.1, 0.1]
   - Interpretation: Values outside this range indicate potential bias

   Findings:
   - Gender SPD: {:.4f} - {}
   - Race SPD: {:.4f} - {}
   - Age SPD: {:.4f} - {}

2. DISPARATE IMPACT RATIO (DIR):
   - Measures: Ratio of approval rates (4/5 rule compliance)
   - Minimum Acceptable: 0.80
   - Interpretation: Values < 0.80 suggest discriminatory impact

   Findings:
   - Gender DIR: {:.4f} - {}
   - Race DIR: {:.4f} - {}
   - Age DIR: {:.4f} - {}

3. EQUAL OPPORTUNITY DIFFERENCE (EOD):
   - Measures: Gap in True Positive Rates between groups
   - Maximum Acceptable: 0.1
   - Interpretation: Higher values indicate unequal opportunity

   Findings:
   - Gender EOD: {:.4f} - {}
   - Race EOD: {:.4f} - {}
   - Age EOD: {:.4f} - {}

RECOMMENDATIONS FOR IMPROVING FAIRNESS:
1. Data-Level Interventions:
   - Collect more representative data from underrepresented groups
   - Review data collection methods for potential biases
   - Check for and correct any systematic errors

2. Pre-Processing:
   - Use stratified sampling to balance representation
   - Apply fair feature engineering techniques
   - Remove or handle sensitive attributes appropriately

3. In-Processing:
   - Implement fairness constraints during model training
   - Use regularization techniques to penalize biased predictions
   - Consider ensemble methods that balance accuracy and fairness

4. Post-Processing:
   - Adjust decision thresholds per group to achieve fairness
   - Implement acceptance/rejection policies that respect fairness constraints
   - Use calibration techniques to equalize error rates across groups

5. Monitoring:
   - Regularly audit model predictions for fairness
   - Track fairness metrics over time
   - Implement feedback loops to detect and correct emerging biases
""".format(
    spd_gender, "✓ FAIR" if abs(spd_gender) <= 0.1 else "✗ BIASED",
    spd_race, "✓ FAIR" if abs(spd_race) <= 0.1 else "✗ BIASED",
    spd_age, "✓ FAIR" if abs(spd_age) <= 0.1 else "✗ BIASED",
    dir_gender, "✓ FAIR" if dir_gender >= 0.80 else "✗ BIASED",
    dir_race, "✓ FAIR" if dir_race >= 0.80 else "✗ BIASED",
    dir_age, "✓ FAIR" if dir_age >= 0.80 else "✗ BIASED",
    eod_gender, "✓ FAIR" if eod_gender < 0.1 else "✗ BIASED",
    eod_race, "✓ FAIR" if eod_race < 0.1 else "✗ BIASED",
    eod_age, "✓ FAIR" if eod_age < 0.1 else "✗ BIASED"
)

print(findings)

print("\n" + "=" * 80)
print("ANALYSIS COMPLETE")
print("=" * 80)


In [ ]:
import pandas as pd
import numpy as np

# AIF360 metrics (sklearn wrappers)
from aif360.sklearn.metrics import (
    statistical_parity_difference,
    disparate_impact_ratio,
    equal_opportunity_difference,
    generalized_fnr,
    generalized_fpr
)

# ---------------------------
# Safe variable resolution
# ---------------------------
# Accept several common variable name variants used earlier
# If none exist, create a small dummy dataset (for quick testing)
def get_var(*names, default=None):
    for n in names:
        if n in globals():
            return globals()[n]
    return default

# Try to find existing series/arrays in the notebook's namespace
y_true_series = get_var('y_test_series', 'y_test_reset', 'y_test', default=None)
y_pred_series = get_var('y_pred_series', 'y_pred_reset', 'y_pred', default=None)
sex_series   = get_var('sex_test', 'sex_test_reset', default=None)
race_series  = get_var('race_test', 'race_test_reset', default=None)
age_arr      = get_var('age_test', 'age_test_reset', default=None)

# If any core variable is missing, create a small dummy example and warn
if y_true_series is None or y_pred_series is None or sex_series is None or race_series is None or age_arr is None:
    print("Warning: Some input variables not found. Using a small dummy dataset for testing.")
    y_true_series = pd.Series([1,1,0,1,0,1,0,1,1,0])
    y_pred_series = pd.Series([1,0,0,1,0,1,0,0,1,0])
    sex_series = pd.Series(['Male','Female','Female','Male','Female','Male','Female','Male','Female','Male'])
    race_series = pd.Series(['White','Black','White','Black','Asian','White','Black','White','Asian','White'])
    age_arr = pd.Series([25, 42, 22, 35, 60, 45, 29, 33, 52, 48])

# Convert to pandas Series (if arrays) and reset indices
y_true_series = pd.Series(y_true_series).reset_index(drop=True)
y_pred_series = pd.Series(y_pred_series).reset_index(drop=True)
sex_series    = pd.Series(sex_series).reset_index(drop=True)
race_series   = pd.Series(race_series).reset_index(drop=True)
age_arr       = pd.Series(age_arr).reset_index(drop=True)

# ---------------------------
# Create consistent age groups
# ---------------------------
# Use same bins & labels as you intended
age_bins = [0, 30, 50, 150]
age_labels = ['Young (<30)', 'Middle-aged (30-50)', 'Senior (>50)']
age_categories_test = pd.cut(age_arr.astype(float), bins=age_bins, labels=age_labels, include_lowest=True)

# ---------------------------
# Build the protected DataFrame
# ---------------------------
test_df_protected = pd.DataFrame({
    'y_true': y_true_series.values,
    'y_pred': y_pred_series.values,
    'sex': sex_series.values,
    'race': race_series.values,
    'age_group': age_categories_test.values
})

# Drop any rows with NaNs produced during binning/creation
test_df_protected.dropna(inplace=True)
test_df_protected['sex'] = test_df_protected['sex'].astype(str)
test_df_protected['race'] = test_df_protected['race'].astype(str)
test_df_protected['age_group'] = test_df_protected['age_group'].astype(str)

# Create intersectional key
test_df_protected['intersectional_group'] = (
    test_df_protected['sex'] + '_' +
    test_df_protected['race'] + '_' +
    test_df_protected['age_group']
)

print(f"Total test samples for intersectional analysis: {len(test_df_protected)}")
print(f"Number of unique intersectional groups: {test_df_protected['intersectional_group'].nunique()}")

# ---------------------------
# Privileged baseline selection
# ---------------------------
# Try the exact privileged label you provided; otherwise, attempt to auto-find a sensible privileged group
preferred_privileged = 'Male_White_Middle-aged (30-50)'
groups = test_df_protected['intersectional_group'].unique().tolist()

def find_best_privileged(preferred, groups):
    if preferred in groups:
        return preferred
    # try alternatives (some code paths created age labels without parentheses)
    alt = preferred.replace(' (30-50)', '(30-50)')
    if alt in groups:
        return alt
    # Try to find a group that contains 'Male' and 'White' and 'Middle' heuristically
    for g in groups:
        if 'Male' in g and 'White' in g and ('Middle' in g or '30-50' in g):
            return g
    # fallback: largest group
    return test_df_protected['intersectional_group'].value_counts().idxmax()

privileged_intersectional_group = find_best_privileged(preferred_privileged, groups)

if privileged_intersectional_group not in groups:
    print(f"Warning: Could not find specified privileged group. Using fallback: {privileged_intersectional_group}")
else:
    print(f"Using privileged baseline: {privileged_intersectional_group}")

# ---------------------------
# Compute privileged group stats
# ---------------------------
privileged_group_data = test_df_protected[test_df_protected['intersectional_group'] == privileged_intersectional_group]
n_priv = len(privileged_group_data)
y_true_priv = privileged_group_data['y_true'].values if n_priv > 0 else np.array([])
y_pred_priv = privileged_group_data['y_pred'].values if n_priv > 0 else np.array([])

priv_approval_rate = np.nan
priv_tpr = np.nan
priv_fpr = np.nan
if n_priv > 0:
    priv_approval_rate = np.mean(y_pred_priv)
    if (y_true_priv == 1).sum() > 0:
        priv_tpr = np.mean(y_pred_priv[y_true_priv == 1])
    if (y_true_priv == 0).sum() > 0:
        priv_fpr = np.mean(y_pred_priv[y_true_priv == 0])

print("\n" + "="*60)
print(f"Privileged Baseline: {privileged_intersectional_group} (n={n_priv})")
print("="*60)
print(f"  Privileged Approval Rate: {np.nan if np.isnan(priv_approval_rate) else f'{priv_approval_rate:.4f}'}")
print(f"  Privileged TPR:           {np.nan if np.isnan(priv_tpr) else f'{priv_tpr:.4f}'}")
print(f"  Privileged FPR:           {np.nan if np.isnan(priv_fpr) else f'{priv_fpr:.4f}'}")
print("-"*60)

# ---------------------------
# Compute metrics for each intersectional group vs privileged baseline
# ---------------------------
intersectional_metrics = {}
min_group_size = 10  # threshold for reliable metrics (adjustable)

for group, group_df in test_df_protected.groupby('intersectional_group'):
    if group == privileged_intersectional_group:
        continue
    n_group = len(group_df)
    if n_group < min_group_size or n_priv < min_group_size:
        # Skip groups that are too small for reliable estimates
        intersectional_metrics[group] = {
            'n_samples': n_group,
            'spd': np.nan, 'dir': np.nan, 'eod': np.nan, 'fnr_diff': np.nan, 'fpr_diff': np.nan,
            'note': 'small_sample'
        }
        continue

    y_true_group = group_df['y_true'].values
    y_pred_group = group_df['y_pred'].values

    # Combined arrays: privileged first (label 0), current group second (label 1)
    combined_y_true = np.concatenate([y_true_priv, y_true_group])
    combined_y_pred = np.concatenate([y_pred_priv, y_pred_group])
    combined_prot_attr = np.concatenate([np.zeros(n_priv, dtype=int), np.ones(n_group, dtype=int)])

    # compute metrics with safe try/except
    try:
        spd = statistical_parity_difference(combined_y_true, combined_y_pred,
                                           prot_attr=combined_prot_attr, priv_group=0, pos_label=1)
    except Exception as e:
        spd = np.nan

    try:
        dir_score = disparate_impact_ratio(combined_y_true, combined_y_pred,
                                           prot_attr=combined_prot_attr, priv_group=0, pos_label=1)
    except Exception as e:
        dir_score = np.nan

    try:
        eod = equal_opportunity_difference(combined_y_true, combined_y_pred,
                                           prot_attr=combined_prot_attr, priv_group=0, pos_label=1)
    except Exception as e:
        eod = np.nan

    try:
        fnr_diff = generalized_fnr(combined_y_true, combined_y_pred,
                                   prot_attr=combined_prot_attr, priv_group=0, pos_label=1)
    except Exception as e:
        fnr_diff = np.nan

    try:
        fpr_diff = generalized_fpr(combined_y_true, combined_y_pred,
                                   prot_attr=combined_prot_attr, priv_group=0, pos_label=1)
    except Exception as e:
        fpr_diff = np.nan

    intersectional_metrics[group] = {
        'n_samples': n_group,
        'spd': spd,
        'dir': dir_score,
        'eod': eod,
        'fnr_diff': fnr_diff,
        'fpr_diff': fpr_diff,
        'note': ''
    }

    # print concise results for the group
    print(f"\nGroup: {group} (n={n_group})")
    print(f"  SPD:       {spd if not np.isnan(spd) else 'N/A'}")
    print(f"  DIR:       {dir_score if not np.isnan(dir_score) else 'N/A'}")
    print(f"  EOD:       {eod if not np.isnan(eod) else 'N/A'}")
    print(f"  FNR Diff:  {fnr_diff if not np.isnan(fnr_diff) else 'N/A'}")
    print(f"  FPR Diff:  {fpr_diff if not np.isnan(fpr_diff) else 'N/A'}")

# ---------------------------
# Produce a tidy results DataFrame
# ---------------------------
results_df = pd.DataFrame.from_dict(intersectional_metrics, orient='index').reset_index().rename(columns={'index': 'intersectional_group'})
# Add columns for easier reading
results_df = results_df[['intersectional_group', 'n_samples', 'spd', 'dir', 'eod', 'fnr_diff', 'fpr_diff', 'note']]

# Sort by sample size desc and then by SPD magnitude (abs)
results_df['abs_spd'] = results_df['spd'].abs()
results_df = results_df.sort_values(['n_samples', 'abs_spd'], ascending=[False, False]).drop(columns=['abs_spd'])

print("\n" + "="*60)
print("Top intersectional groups summary (first 20 rows):")
print("="*60)
print(results_df.head(20).to_string(index=False))

print("\n✓ Intersectional fairness analysis complete")
